# Урок 5. Динамическое программирование: знакомство

**11 класс · I четверть · неделя 5 · 40 минут**

🐍 Python  🎓 ЕГЭ

[⬆ Оглавление 11 класса](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 4](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-04.ipynb) · [Урок 6 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-06.ipynb)

---

### Что будет на уроке

Задачи с накоплением ответа: числа Фибоначчи, лестница, маршруты в таблице. Таблица подзадач вместо рекурсии.

| Этап | Время |
|---|---|
| Разбор теории | 15 мин |
| Примеры с разбором | 10 мин |
| Задачи в классе | 12 мин |
| Домашнее задание | 3 мин |

In [ ]:
#@title 🚀 Шаг 1. Регистрация и подготовка урока { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
ФИО = "" #@param {type:"string"}
Класс = "11А" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-05", name=ФИО, klass=Класс)

## Часть 1. Теория — 15 минут

### Когда перебор не работает

Задача: сколькими способами можно подняться по лестнице из n ступенек,
если за шаг можно перешагнуть через одну или две ступеньки?

Прямолинейное решение — рекурсия: чтобы попасть на ступеньку n,
нужно прийти либо с n−1, либо с n−2.

```python
def способов(n):
    if n <= 1:
        return 1
    return способов(n - 1) + способов(n - 2)
```

Работает — и катастрофически медленно. Для n = 40 программа задумается
на минуты, для n = 50 не закончит никогда.

Причина видна на дереве вызовов: `способов(5)` вызывает `способов(4)`
и `способов(3)`; `способов(4)` снова вызывает `способов(3)`.
**Одни и те же подзадачи решаются заново тысячи раз.**

```
                 f(5)
             ╱          ╲
         f(4)            f(3)
        ╱    ╲          ╱    ╲
     f(3)    f(2)    f(2)    f(1)
    ╱   ╲    ╱  ╲    ╱  ╲
  f(2) f(1) …    …  …    …
```

### Идея динамического программирования

Решение простое до неприличия: **запомнить ответы**.

Вместо рекурсии заводим таблицу и заполняем её от простых случаев
к сложным:

```python
способов = [0] * (n + 1)
способов[0] = 1
способов[1] = 1
for i in range(2, n + 1):
    способов[i] = способов[i - 1] + способов[i - 2]
```

Каждая подзадача решается **ровно один раз**. Сложность падает
с экспоненциальной до линейной.

Этот приём и называется **динамическим программированием**.
Название историческое и не очень удачное — речь просто о таблице
промежуточных результатов.

### Когда приём применим

Два признака, оба обязательны.

**Оптимальная подструктура.** Ответ для задачи выражается через ответы
для меньших задач. В примере с лестницей: способов на ступеньку n
равно сумме способов на n−1 и n−2.

**Перекрывающиеся подзадачи.** Одни и те же меньшие задачи возникают
многократно. Если бы каждая подзадача встречалась один раз,
запоминать было бы нечего.

### Схема решения

1. **Определить состояние** — что именно хранится в ячейке таблицы.
2. **Записать переход** — как ячейка выражается через предыдущие.
3. **Задать базу** — с каких значений начинаем.
4. **Выбрать порядок заполнения** — от базы к ответу.

### Классические задачи

**Числа Фибоначчи.** $F_n = F_{n-1} + F_{n-2}$, база $F_0 = 0$, $F_1 = 1$.

**Лестница.** То же соотношение, другая база.

**Маршруты в таблице.** Робот идёт из левого верхнего угла в правый
нижний, двигаясь только вправо и вниз. Количество путей в клетку
равно сумме путей сверху и слева. Задача решается **той же расстановкой
чисел**, что мы делали для графов в 9 классе.

**Наибольшая сумма пути.** Вместо количества путей копим максимальную
сумму — меняется только операция в переходе.

### Мемоизация: рекурсия с памятью

Есть и второй способ — оставить рекурсию, но запоминать результаты
в словаре:

```python
память = {}

def способов(n):
    if n <= 1:
        return 1
    if n not in память:
        память[n] = способов(n - 1) + способов(n - 2)
    return память[n]
```

Эффект тот же, код ближе к исходной формулировке. В Python есть
готовый инструмент — декоратор `functools.lru_cache`, который делает
это автоматически.

## Часть 2. Примеры с разбором — 10 минут

### Пример 1. Цена наивной рекурсии

In [ ]:
import time

вызовов = 0


def наивно(n):
    global вызовов
    вызовов += 1
    if n <= 1:
        return 1
    return наивно(n - 1) + наивно(n - 2)


def таблицей(n):
    таблица = [0] * (n + 1)
    таблица[0] = 1
    if n >= 1:
        таблица[1] = 1
    for i in range(2, n + 1):
        таблица[i] = таблица[i - 1] + таблица[i - 2]
    return таблица[n]


for n in [10, 20, 30]:
    вызовов = 0
    начало = time.perf_counter()
    ответ = наивно(n)
    t1 = time.perf_counter() - начало
    сделано = вызовов

    начало = time.perf_counter()
    таблицей(n)
    t2 = time.perf_counter() - начало

    print(f"n = {n:>2}: ответ {ответ:>8}, "
          f"наивно {сделано:>8} вызовов ({t1 * 1000:>7.1f} мс), "
          f"таблицей {t2 * 1000:.3f} мс")

Количество вызовов растёт примерно вдвое с каждым увеличением n
на единицу — это и есть экспоненциальная сложность. Таблица же
делает ровно n шагов.

Для n = 30 разница уже в тысячи раз, для n = 50 наивный способ
просто не завершится.

### Пример 2. Маршруты робота

In [ ]:
def маршрутов(строк, столбцов, препятствия=()):
    таблица = [[0] * столбцов for _ in range(строк)]

    for i in range(строк):
        for j in range(столбцов):
            if (i, j) in препятствия:
                таблица[i][j] = 0
            elif i == 0 and j == 0:
                таблица[i][j] = 1
            else:
                сверху = таблица[i - 1][j] if i > 0 else 0
                слева = таблица[i][j - 1] if j > 0 else 0
                таблица[i][j] = сверху + слева

    return таблица


поле = маршрутов(4, 4)
print("Количество путей в каждую клетку (поле 4×4):")
for строка in поле:
    print("  " + " ".join(f"{значение:>4}" for значение in строка))
print(f"\nВсего путей в правый нижний угол: {поле[-1][-1]}")

поле2 = маршрутов(4, 4, препятствия={(1, 1), (2, 2)})
print("\nТо же поле с двумя препятствиями:")
for строка in поле2:
    print("  " + " ".join(f"{значение:>4}" for значение in строка))
print(f"\nПутей осталось: {поле2[-1][-1]}")

Заметьте, насколько это похоже на подсчёт путей в графе из 9 класса:
та же расстановка чисел, то же правило «сумма всех, откуда можно
прийти». Динамическое программирование — обобщение того приёма.

Препятствия обрабатываются одной строкой: в такой клетке путей ноль,
и она автоматически перестаёт передавать что-либо дальше.

### Пример 3. Максимальная сумма пути

Меняем одну операцию — и вместо количества путей получаем оптимум.

In [ ]:
поле_ценностей = [
    [1, 3, 1, 2],
    [1, 5, 1, 3],
    [4, 2, 1, 4],
    [1, 1, 8, 1],
]

строк, столбцов = len(поле_ценностей), len(поле_ценностей[0])
лучшее = [[0] * столбцов for _ in range(строк)]

for i in range(строк):
    for j in range(столбцов):
        сверху = лучшее[i - 1][j] if i > 0 else -1
        слева = лучшее[i][j - 1] if j > 0 else -1
        предыдущее = max(сверху, слева, 0 if i == 0 and j == 0 else -1)
        лучшее[i][j] = поле_ценностей[i][j] + max(предыдущее, 0)

print("Поле ценностей:")
for строка in поле_ценностей:
    print("  " + " ".join(f"{з:>3}" for з in строка))

print("\nМаксимальная сумма до каждой клетки:")
for строка in лучшее:
    print("  " + " ".join(f"{з:>3}" for з in строка))

print(f"\nМаксимальная сумма пути: {лучшее[-1][-1]}")

Единственное отличие от предыдущего примера — вместо сложения
путей мы берём **максимум** из возможных предшественников
и прибавляем ценность текущей клетки.

Одна и та же схема, разные операции в переходе: сложение даёт
количество, максимум даёт оптимум. Это очень типично для задач ЕГЭ.

## Часть 3. Задачи в классе — 12 минут

### Задача 1. Лестница

Сколькими способами можно подняться на n ступенек, если за шаг
можно преодолеть 1 или 2 ступеньки?

Обязательно таблицей — рекурсия не пройдёт по времени.

База: `способов(0) = 1`, `способов(1) = 1`.

In [ ]:
def лестница(n):
    return ...

In [ ]:
si.check("1", лестница, [
    (0, 1),
    (1, 1),
    (2, 2),
    (5, 8),
    (10, 89),
    (50, 20365011074),
])

### Задача 2. Лестница с тремя шагами

То же самое, но шагнуть можно на 1, 2 или 3 ступеньки.

Переход меняется на сумму трёх предыдущих. Не забудьте расширить базу.

In [ ]:
def лестница3(n):
    return ...

In [ ]:
si.check("2", лестница3, [
    (0, 1),
    (1, 1),
    (2, 2),
    (3, 4),
    (5, 13),
    (10, 274),
])

### Задача 3. Маршруты в таблице

Сколько путей ведёт из левого верхнего угла в правый нижний,
если двигаться можно только вправо и вниз?

In [ ]:
def путей(строк, столбцов):
    return ...

In [ ]:
si.check("3", путей, [
    ((1, 1), 1),
    ((2, 2), 2),
    ((3, 3), 6),
    ((4, 4), 20),
    ((1, 10), 1),
])

## Часть 4. Домашнее задание

### Домашнее задание 1. Кузнечик

Классическая задача ЕГЭ.

> Кузнечик прыгает по числовой прямой вправо на 1 или на 3.
> Сколько существует различных маршрутов из точки 0 в точку n?

Переход: в точку n можно попасть из n−1 или n−3.

In [ ]:
def кузнечик(n):
    return ...

In [ ]:
si.check("дз1", кузнечик, [
    (0, 1),
    (1, 1),
    (2, 1),
    (3, 2),
    (6, 6),
    (10, 28),
])

### Домашнее задание 2. Кузнечик с запретом

Усложнённая версия: некоторые точки запрещены, через них проходить
нельзя.

`кузнечик_с_запретом(6, [2, 4])` — точки 2 и 4 закрыты.

В запрещённой точке количество маршрутов равно нулю — она просто
перестаёт передавать пути дальше.

In [ ]:
def кузнечик_с_запретом(n, запрещённые):
    return ...

In [ ]:
si.check("дз2", кузнечик_с_запретом, [
    ((6, [2, 4]), 1),
    ((6, []), 6),
    ((5, [1, 2, 3]), 0),
    ((10, [5]), 12),
])

### Домашнее задание 3. Максимальная сумма пути

По таблице чисел верните максимальную сумму, которую можно собрать
на пути из левого верхнего угла в правый нижний, двигаясь только
вправо и вниз.

Схема та же, что в примере 3: вместо сложения путей берём максимум
из предшественников.

In [ ]:
def максимальный_путь(поле):
    return ...

In [ ]:
si.check("дз3", максимальный_путь, [
    ([[1, 3, 1], [1, 5, 1], [4, 2, 1]], 12),
    ([[5]], 5),
    ([[1, 2], [3, 4]], 8),
    ([[1, 1, 1], [1, 1, 1]], 4),
])

---

### Как узнать задачу на динамическое программирование

Три признака, по которым её видно почти всегда:

1. спрашивают **количество способов** или **оптимум** (максимум,
   минимум, наименьшая стоимость);
2. ответ для n естественно выражается через ответы для меньших
   значений;
3. наивный перебор явно не уложится во время.

Если все три сходятся — заводите таблицу. В заданиях ЕГЭ 25–27
это самый частый работающий подход.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 4](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-04.ipynb) · [⬆ Оглавление 11 класса](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 6 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-06.ipynb)